# Leak-free real-estate training with MLflow

This notebook is a review-friendly entry point. Production logic lives in the tested `mlflow_project` package, so the registered model contains the complete raw-feature pipeline. The data policy is fixed: **60% train, 20% validation, 20% test**. Test labels are accessed once, after model selection and final refit.

In [ ]:
from dataclasses import asdict

import pandas as pd

from mlflow_project.config import AppConfig
from mlflow_project.data import load_dataset, split_dataset
from mlflow_project.features import RealEstateFeatureBuilder
from mlflow_project.training import train_and_register

## 1. Configuration and safe SQL extraction

`AppConfig.from_env()` loads the repository `.env`. SQLAlchemy reflects the validated `schema.table` identifier and selects an explicit, versioned column contract.

In [ ]:
config = AppConfig.from_env()
data = load_dataset(
    config.database,
    config.table_name,
    target_column=config.target_column,
)
splits = split_dataset(
    data,
    target_column=config.target_column,
    validation_size=config.validation_size,
    test_size=config.test_size,
    random_state=config.random_state,
    group_column=config.split_group_column,
)
splits.assert_disjoint()
splits.summary()

## 2. Training-only data review

Target statistics are calculated on the training partition only. The validation and test targets remain outside exploratory decisions.

In [ ]:
train_frame = splits.X_train.assign(**{config.target_column: splits.y_train})
pd.DataFrame({
    'dtype': train_frame.dtypes.astype(str),
    'missing': train_frame.isna().sum(),
    'unique': train_frame.nunique(dropna=False),
})

In [ ]:
feature_builder = RealEstateFeatureBuilder(reference_year=config.reference_year)
feature_preview = feature_builder.fit_transform(splits.X_train.head(10))
feature_preview.T

## 3. Tune, evaluate once and register

Optuna and `RandomizedSearchCV` fit on train and score on the separate validation partition. The winner is refit on train+validation. The final test metrics and the complete sklearn pipeline are then logged to MLflow, and only the model version belonging to the current run is tagged.

In [ ]:
result = train_and_register(data, splits, config)
asdict(result)